In [ ]:
!pip install pymupdf gradio openai langchain langchain-openai faiss-cpu google-colab langchain_community



In [ ]:
import os
import pymupdf
import gradio as gr
import openai
from langchain_openai import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.docstore.document import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Set OpenAI API Key (Using OpenRouter API)
os.environ["OPENAI_API_KEY"] = "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"  # Replace with actual key
OPENROUTER_API_BASE = "https://openrouter.ai/api/v1"

# Path to PDF file in Drive
PDF_PATH = "/content/12th_General English_Text_www.tntextbooks.in (1).pdf"  # Update with actual path


def extract_text_from_pdf(pdf_path):
    """Extracts text from a PDF file."""
    doc = pymupdf.open(pdf_path)
    return "\n".join([page.get_text("text") for page in doc])


def initialize_embeddings():
    """Initializes OpenAI embeddings."""
    return OpenAIEmbeddings(openai_api_key=os.getenv("OPENAI_API_KEY"))


def create_faiss_index(docs):
    """Creates FAISS index from document chunks."""
    embeddings = initialize_embeddings()
    vector_store = FAISS.from_documents(docs, embeddings)
    vector_store.save_local("faiss_index")
    return vector_store


def load_faiss_index():
    """Loads FAISS index from storage, if available."""
    embeddings = initialize_embeddings()
    if os.path.exists("faiss_index"):
        return FAISS.load_local("faiss_index", embeddings)
    return None


def query_chatbot(query):
    """Queries the chatbot using FAISS and OpenAI."""
    vector_store = load_faiss_index()
    if not vector_store:
        return "FAISS index not found. Please process the PDF first."

    docs = vector_store.similarity_search(query, k=3)
    context = "\n".join([doc.page_content for doc in docs])

    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        api_base=OPENROUTER_API_BASE,
        api_key=os.getenv("OPENAI_API_KEY"),
        messages=[
            {"role": "system", "content": "You are a helpful assistant that answers based on the provided documents."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuery: {query}"}
        ]
    )
    return response["choices"][0]["message"]["content"]

# Process PDF and create FAISS index
pdf_text = extract_text_from_pdf(PDF_PATH)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = [Document(page_content=chunk) for chunk in text_splitter.split_text(pdf_text)]
vector_store = create_faiss_index(docs)

# Gradio UI
with gr.Blocks() as ui:
    gr.Markdown("# 📚 PDF Chatbot 🤖")

    with gr.Row():
        user_input = gr.Textbox(label="Ask a question about the document:")
        submit_btn = gr.Button("🔍 Search")

    chat_output = gr.Textbox(label="🤖 Chatbot Response", interactive=False)

    submit_btn.click(query_chatbot, inputs=user_input, outputs=chat_output)

# Launch the app
ui.launch()
